# Dragonfly Single-Node Effect Study
This notebook runs a single-node dragonfly parameter study and estimates which topology variable has the strongest effect on execution phase duration.

## 0. User Parameters

In [1]:
import os

# Path to the benchmark workflow directory
workflow_dir = '/lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark/'

# Give this study its own output namespace
experiment_name = 'dragonfly_effect_study'
output_dir = os.path.join(workflow_dir, experiment_name + '_out')
experiment_result_file = os.path.join(workflow_dir, f'{experiment_name}_results.csv')

# Container and tooling paths
apptainer_bin_path = '/lus/scratch/crickett/software/usr/bin'
apptainer_cache_dir = '/lus/bnchlu1/neth/.local/apptainer/'
sst_container_url = 'ghcr.io/hpc-ai-adv-dev/sst-perf-track-full:16.0.0'
sst_container_dest_name = 'sst-perf-track-full.sif'
sst_container_dest_path = os.path.join(workflow_dir, sst_container_dest_name)
sst_input_config = os.path.join(workflow_dir, 'merlin_benchmark.py')

print('workflow_dir:', workflow_dir)
print('output_dir:', output_dir)
print('result_file:', experiment_result_file)

workflow_dir: /lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark/
output_dir: /lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark/dragonfly_effect_study_out
result_file: /lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark/dragonfly_effect_study_results.csv


## 1. Imports and Environment

In [2]:
from workflow_utils import *
import sys
import json
import glob
import humanfriendly
import pandas as pd
from plotnine import *

sys.path.append(workflow_dir)
import workflow_args

os.environ['PATH'] += os.pathsep + apptainer_bin_path
os.environ['LANG'] = 'C.UTF-8'
os.environ['LC_ALL'] = 'C.UTF-8'
os.environ['APPTAINER_CACHEDIR'] = apptainer_cache_dir

## 2. One-Time Setup (Container, Profile, Build)
Run this once per environment; skip if already prepared.

In [6]:
cd(workflow_dir)

get_convert_to_sif()

# script expects there not to be an extension on the output file name
if os.path.exists(sst_container_dest_path):
    print(f"{sst_container_dest_path} already exists.")
else:
    run_cmd(f"./convert-to-sif.sh {sst_container_url} -f {sst_container_dest_path.replace('.sif', '')}")


run_cmd(f'e4s-cl profile edit --add-files {workflow_dir}')
run_cmd(f'e4s-cl profile edit --add-files {workflow_dir}/*.py')
run_cmd(f'e4s-cl profile edit --image {sst_container_dest_path}')

run_cmd('touch sstsimulator.conf')
run_in_container('make -j10', sst_container_dest_path, additional_apptainer_args=f'--bind sstsimulator.conf:{os.getenv("HOME")}/.sst/sstsimulator.conf')

> cd /lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark/
/lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark/sst-perf-track-full.sif already exists.
> e4s-cl profile edit --add-files /lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark/


File /lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark already in profile's files


> e4s-cl profile edit --add-files /lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark//*.py


File /lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark/*.py already in profile's files


> e4s-cl profile edit --image /lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark/sst-perf-track-full.sif
> touch sstsimulator.conf
> ['apptainer', 'exec', '--no-home', '--bind', '/lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark:/lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark', '--pwd', '/lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark', '--bind', 'sstsimulator.conf:/home/users/neth/.sst/sstsimulator.conf', '/lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark/sst-perf-track-full.sif', 'bash', '-lc', 'make -j10']
/opt/SST/16.0.0/bin//sst-register merlin_benchmark merlin_benchmark_LIBDIR=/lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark


## 3. Build Single-Node Dragonfly Sweep

In [8]:
run_specs = workflow_args.generate_merlin_run_specs(
    node_counts=(1,),
    rank_counts=(1,),
    topologies=('dragonfly',),
    global_params={'stop_at': (50)},
    dragonfly_params={
        'dragonfly_num_groups': (8, 16, 32, 64, 128),
        'dragonfly_hosts_per_router': (2, 4, 8, 16, 32),
        'dragonfly_routers_per_group': (4, 8, 16, 32, 64),
    },
    experiment_name=experiment_name,
)

print(f'Total runs to launch: {len(run_specs)}')

Total runs to launch: 125


## 4. Launch Runs

In [9]:
FORCED = 0
os.makedirs(output_dir, exist_ok=True)

for run_spec in run_specs:
    run_output_dir = os.path.join(output_dir, run_spec.run_name)
    os.makedirs(run_output_dir, exist_ok=True)
    cd(run_output_dir)

    status_file_path = os.path.join(run_output_dir, 'status.txt')
    if os.path.exists(status_file_path):
        with open(status_file_path, 'r') as f:
            status = f.read().strip()
        if (not FORCED) and status in ['COMPLETED', 'SUBMITTED']:
            print(f'Skipping {run_spec.run_name} as it is already {status}')
            continue

    log_file = os.path.join(run_output_dir, 'run.log')
    error_file = os.path.join(run_output_dir, 'run.err')
    profiling_output_file = os.path.join(run_output_dir, 'profiling.json')

    with open(os.path.join(run_output_dir, 'params.json'), 'w') as f:
        json.dump(run_spec.to_dict(), f, indent=2)

    srun_part = ' '.join(run_spec.launcher['srun'])
    sst_part = ' '.join(run_spec.sst_args)
    config_part = ' '.join(run_spec.config_args)

    launch_cmd = (
        f'e4s-cl launch srun --output={log_file} --error={error_file} '
        f'{srun_part} -- sst --profiling-output={profiling_output_file} {sst_part} {sst_input_config} -- {config_part}'
    )

    sbatch_script_path = os.path.join(run_output_dir, 'run.sbatch')
    with open(sbatch_script_path, 'w') as f:
        f.write('#!/bin/bash\n')
        f.write(f"echo 'LAUNCHED' > {status_file_path}\n")
        f.write(f'{launch_cmd}\n')
        f.write('if [ $? -eq 0 ]; then\n')
        f.write(f"    echo 'COMPLETED' > {status_file_path}\n")
        f.write('else\n')
        f.write(f"    echo 'FAILED' > {status_file_path}\n")
        f.write('fi\n')
    os.chmod(sbatch_script_path, 0o755)

    sbatch_log = os.path.join(run_output_dir, 'sbatch.log')
    sbatch_err = os.path.join(run_output_dir, 'sbatch.err')
    sbatch_cmd = f'sbatch --job-name={experiment_name} --output={sbatch_log} --error={sbatch_err} {srun_part} {sbatch_script_path}'
    with open(os.path.join(run_output_dir, 'sbatch_submit_cmd.txt'), 'w') as f:
        f.write(f'{sbatch_cmd}\n')

    with open(status_file_path, 'w') as f:
        f.write('SUBMITTED\n')
    run_cmd(sbatch_cmd)

    cd(workflow_dir)

> cd /lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark/dragonfly_effect_study_out/dragonfly_effect_study_dragonfly_n1_r1_edc07e53c1
> sbatch --job-name=dragonfly_effect_study --output=/lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark/dragonfly_effect_study_out/dragonfly_effect_study_dragonfly_n1_r1_edc07e53c1/sbatch.log --error=/lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark/dragonfly_effect_study_out/dragonfly_effect_study_dragonfly_n1_r1_edc07e53c1/sbatch.err --nodes=1 --ntasks-per-node=1 /lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark/dragonfly_effect_study_out/dragonfly_effect_study_dragonfly_n1_r1_edc07e53c1/run.sbatch
Submitted batch job 1585580
> cd /lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark/
> cd /lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark/dragonfly_effect_study_out/dragonfly_effect_study_dragonfly_n1_r1_2adb27e27b
> sbatch --job-name=dragonfly_effect_study --output=/lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark/dragonfly_effect_study_out/dragonfly_effect_st

## 5. Wait and Collect Results

In [10]:
wait_for_jobs(job_name=experiment_name)

result_files = glob.glob(f'{output_dir}/**/profiling.json', recursive=True)
print(f'Found {len(result_files)} profiling files')

rows = []
for result_file in result_files:
    row = {}
    param_file = result_file.replace('profiling.json', 'params.json')

    if os.path.exists(param_file):
        with open(param_file, 'r') as f:
            row.update(json.load(f))

    with open(result_file, 'r') as f:
        content = json.load(f)
        regions = content['regions']
        row['total_duration_s'] = humanfriendly.parse_timespan(regions['total']['duration'])
        row['build_duration_s'] = humanfriendly.parse_timespan(regions['total']['build']['duration'])
        row['execute_duration_s'] = humanfriendly.parse_timespan(regions['total']['execute']['duration'])

    rows.append(row)

df = pd.DataFrame(rows)
df.to_csv(experiment_result_file, index=False)
df.head()

Waiting for jobs to complete... Queue length: 93
Waiting for jobs to complete... Queue length: 69
Waiting for jobs to complete... Queue length: 46
Waiting for jobs to complete... Queue length: 34
Waiting for jobs to complete... Queue length: 24
Waiting for jobs to complete... Queue length: 19
Waiting for jobs to complete... Queue length: 15
Waiting for jobs to complete... Queue length: 14
Waiting for jobs to complete... Queue length: 12
Waiting for jobs to complete... Queue length: 8
Waiting for jobs to complete... Queue length: 8
Waiting for jobs to complete... Queue length: 7
Waiting for jobs to complete... Queue length: 5
Waiting for jobs to complete... Queue length: 5
Waiting for jobs to complete... Queue length: 5
Waiting for jobs to complete... Queue length: 5
Waiting for jobs to complete... Queue length: 4
Waiting for jobs to complete... Queue length: 4
Waiting for jobs to complete... Queue length: 4
Waiting for jobs to complete... Queue length: 4
Waiting for jobs to complete...

,run_id,run_name,experiment_name,topology,node_count,rank_count,srun_command,mpirun_command,stop_at,verbose,...,dragonfly_hosts_per_router,dragonfly_routers_per_group,dragonfly_intergroup_links,dragonfly_num_groups,dragonfly_algorithm,dragonfly_adaptive_threshold,dragonfly_global_routing_mode,total_duration_s,build_duration_s,execute_duration_s
0,0968bd7b7f,dragonfly_effect_study_dragonfly_n1_r1_0968bd7b7f,dragonfly_effect_study,dragonfly,1,1,srun --nodes=1 --ntasks-per-node=1 sst --timin...,mpirun -n 1 -N 1 sst --timing-info=3 --paralle...,50ns,None,...,4,4,1,8,minimal,2.0,absolute,0.492337,0.480652,0.011211
1,ecc5c64f65,dragonfly_effect_study_dragonfly_n1_r1_ecc5c64f65,dragonfly_effect_study,dragonfly,1,1,srun --nodes=1 --ntasks-per-node=1 sst --timin...,mpirun -n 1 -N 1 sst --timing-info=3 --paralle...,50ns,None,...,4,16,1,8,minimal,2.0,absolute,0.792185,0.754898,0.036925
2,75505585a4,dragonfly_effect_study_dragonfly_n1_r1_75505585a4,dragonfly_effect_study,dragonfly,1,1,srun --nodes=1 --ntasks-per-node=1 sst --timin...,mpirun -n 1 -N 1 sst --timing-info=3 --paralle...,50ns,None,...,2,32,1,64,minimal,2.0,absolute,6.919032,5.764453,1.153437
3,c048fc48bb,dragonfly_effect_study_dragonfly_n1_r1_c048fc48bb,dragonfly_effect_study,dragonfly,1,1,srun --nodes=1 --ntasks-per-node=1 sst --timin...,mpirun -n 1 -N 1 sst --timing-info=3 --paralle...,50ns,None,...,2,8,1,16,minimal,2.0,absolute,0.514558,0.491106,0.021638
4,196616e2d2,dragonfly_effect_study_dragonfly_n1_r1_196616e2d2,dragonfly_effect_study,dragonfly,1,1,srun --nodes=1 --ntasks-per-node=1 sst --timin...,mpirun -n 1 -N 1 sst --timing-info=3 --paralle...,50ns,None,...,32,8,1,32,minimal,2.0,absolute,2.766881,2.218481,0.547337


## 6. Estimate Variable Effect on Execution Phase

In [11]:
study_df = pd.read_csv(experiment_result_file).copy()

effect_vars = [
    'dragonfly_num_groups',
    'dragonfly_hosts_per_router',
    'dragonfly_routers_per_group',
]

required_cols = effect_vars + ['execute_duration_s']
missing = [col for col in required_cols if col not in study_df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

for col in effect_vars:
    study_df[col] = pd.to_numeric(study_df[col], errors='coerce')
study_df['execute_duration_s'] = pd.to_numeric(study_df['execute_duration_s'], errors='coerce')
study_df = study_df.dropna(subset=required_cols)

grand_mean = study_df['execute_duration_s'].mean()
total_ss = ((study_df['execute_duration_s'] - grand_mean) ** 2).sum()

summary_rows = []
for var in effect_vars:
    grouped = study_df.groupby(var)['execute_duration_s']
    means = grouped.mean()
    counts = grouped.size()

    between_ss = ((means - grand_mean) ** 2 * counts).sum()
    eta_sq = (between_ss / total_ss) if total_ss > 0 else float('nan')
    span_pct = ((means.max() - means.min()) / grand_mean * 100.0) if grand_mean > 0 else float('nan')

    summary_rows.append({
        'variable': var,
        'eta_sq': eta_sq,
        'levels': int(means.shape[0]),
        'mean_min_s': means.min(),
        'mean_max_s': means.max(),
        'relative_span_pct': span_pct
    })

effect_summary = pd.DataFrame(summary_rows).sort_values('eta_sq', ascending=False).reset_index(drop=True)
effect_summary

,variable,eta_sq,levels,mean_min_s,mean_max_s,relative_span_pct
0,dragonfly_routers_per_group,0.108069,5,0.207307,19.743763,352.166689
1,dragonfly_num_groups,0.098221,5,0.260815,18.998492,337.767808
2,dragonfly_hosts_per_router,0.076977,5,0.910706,17.485050,298.771267


In [12]:
for var in effect_vars:
    display(
        ggplot(study_df, aes(x=var, y='execute_duration_s'))
        + stat_summary(fun_y='mean', geom='line', group=1)
        + stat_summary(fun_y='mean', geom='point', size=2)
        + labs(title=f'Execution Time vs {var}', x=var, y='Mean execute_duration_s')
    )

TypeError: 'mean' is not a callable object

TypeError: 'mean' is not a callable object

TypeError: 'mean' is not a callable object